# ZetaBench — Kaggle Training Notebook

Run SAC or PPO training on a Kaggle T4 GPU (30 GPU hrs/week free).  
Sessions cut off at 12 hours; use **Cell 6** to resume from the last checkpoint.

**Before running:**
1. Enable Internet: *Settings → Internet → On* (required to clone the repo and install packages)
2. Enable GPU: *Settings → Accelerator → GPU T4 x2* (**do not use P100** — it is incompatible with the pre-installed PyTorch ≥2.3)
3. Add your wandb key: *Settings → Secrets → Add → Name: `WANDB_API_KEY`, Value: your key*
4. Run all cells top-to-bottom for a fresh run, or skip Cell 5 and run Cell 6 to resume.

In [ ]:
# Cell 1 — Clone repo
import os

REPO_URL = "https://github.com/badkoubeh/zeta-bench.git"
REPO_DIR = "/kaggle/working/zeta-bench"
RESULTS_DIR = "/kaggle/working/results"

if not os.path.exists(REPO_DIR):
    !git clone --depth 1 {REPO_URL} {REPO_DIR}
else:
    print("Repo already cloned; pulling latest.")
    !git -C {REPO_DIR} pull --ff-only

os.makedirs(RESULTS_DIR, exist_ok=True)

if not os.path.isdir(REPO_DIR):
    raise RuntimeError(
        f"Clone failed — '{REPO_DIR}' does not exist.\n"
        "Enable internet access: Settings → Internet → On, then re-run this cell."
    )

# Make the repo the working directory so Hydra finds configs/
os.chdir(REPO_DIR)
print("Working directory:", os.getcwd())

In [ ]:
# Cell 2 — Install dependencies
# Kaggle ships a large pre-installed base image (torch, numpy, matplotlib, pandas, etc.).
# Installing the full requirements.lock here causes widespread version conflicts because
# the lockfile was generated for a *clean* Linux environment, not Kaggle's base.
# Instead we install only the packages our project needs that aren't already present,
# then install the package itself in editable mode without pulling transitive deps again.
!pip install -q \
    "gymnasium>=1.2.3" \
    "stable-baselines3>=2.3" \
    "hydra-core>=1.3" \
    "omegaconf>=2.3" \
    "wandb>=0.17" \
    "imageio>=2.34" \
    "imageio-ffmpeg>=0.5"
!pip install -q -e . --no-deps

In [ ]:
# Cell 3 — Verify GPU
# Training uses compute=small_gpu which sets device=cuda. If no GPU is
# detected, warn clearly rather than silently falling back to CPU.
# PyTorch ≥2.3 dropped support for sm_60 (Pascal / P100); minimum is sm_70 (Volta / T4+).
import torch

MIN_SM = 70  # PyTorch ≥2.3 requires sm_70+; sm_60 (P100) is no longer supported

if not torch.cuda.is_available():
    print(
        "WARNING: No GPU detected — training will run on CPU (~20–50x slower).\n"
        "To enable GPU: Settings → Accelerator → GPU T4 x2, then restart and re-run."
    )
else:
    gpu = torch.cuda.get_device_name(0)
    mem_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    major, minor = torch.cuda.get_device_capability(0)
    sm = major * 10 + minor  # e.g. 6.0 → 60, 7.5 → 75

    if sm < MIN_SM:
        raise RuntimeError(
            f"GPU '{gpu}' has CUDA compute capability sm_{sm}, but this PyTorch build "
            f"requires sm_{MIN_SM}+.\n"
            "Fix: Settings → Accelerator → GPU T4 x2 (sm_75), then restart the session and re-run all cells."
        )

    print(f"GPU ready: {gpu} ({mem_gb:.1f} GB, sm_{sm})")

In [ ]:
# Cell 4 — Configure wandb
# The key is pulled from Kaggle Secrets (no hardcoded credentials).
from kaggle_secrets import UserSecretsClient
import os

try:
    secret = UserSecretsClient().get_secret("WANDB_API_KEY")
    os.environ["WANDB_API_KEY"] = secret
    os.environ["WANDB_MODE"] = "online"
    print("wandb: online mode, key loaded from Kaggle Secrets")
except Exception:
    os.environ["WANDB_MODE"] = "offline"
    print("wandb: offline mode (WANDB_API_KEY secret not found)")

In [ ]:
# Cell 5 — Fresh training run
# Adjust agent (sac|ppo), seed, and total_steps as needed.
# Checkpoints save every 50k steps to /kaggle/working/results/ (persists as Kaggle output).
# If the session is cut off before training finishes, use Cell 6 to resume.
# kaggle_gpu profile: n_envs=16, batch_size=1024, gradient_steps=-1 (16 grad steps per rollout).

AGENT = "sac"      # sac | ppo
SEED  = 42
STEPS = 2_000_000

!python experiments/train.py \
    agent={AGENT} \
    seed={SEED} \
    total_steps={STEPS} \
    compute=kaggle_gpu \
    results_dir={RESULTS_DIR}/{AGENT}_moderate_nominal_{SEED}

In [ ]:
# Cell 6 — Resume from latest checkpoint
# Run this cell (instead of Cell 5) at the start of a new session to continue
# a run that was interrupted. Point CHECKPOINT_PATH to the last .zip saved in
# /kaggle/working/results/ (add the previous session's output as a Kaggle dataset
# and mount it, or keep results inside /kaggle/working/ across sessions).

import glob
import os

AGENT = "sac"      # must match the original run
SEED  = 42
STEPS = 2_000_000  # same total; training continues for the remaining steps

run_dir = f"{RESULTS_DIR}/{AGENT}_moderate_nominal_{SEED}"

# Pick the highest-step checkpoint automatically
checkpoints = sorted(glob.glob(f"{run_dir}/*.zip"))
if not checkpoints:
    raise FileNotFoundError(
        f"No checkpoints found in {run_dir}. "
        "Add the previous session's output as a dataset and mount it."
    )
latest = checkpoints[-1]
print(f"Resuming from: {latest}")

!python experiments/train.py \
    agent={AGENT} \
    seed={SEED} \
    total_steps={STEPS} \
    compute=kaggle_gpu \
    resume_from={latest} \
    results_dir={run_dir}